# Genizah focus v1.8 — vision-LoRA A/B on repaired training data

Warm start: **v1.7 step-800** (`isaacmg/qwen3-vl-8b-hebrew-v17-ckpt` @ `068a2cc1`,
eval_loss 0.7805 still descending). Two arms, run **A then B** (one variable per step):

- **Arm A (control)**: repaired data (`genizah_clean_v2`), vision LoRA still frozen,
  700 steps — isolates data-repair + schedule-completion.
- **Arm B (candidate)**: same + `TRAIN_VISION_LORA=True` → the patch_embed
  gradient fix actually trains the vision tower for the first time.
  B − A at matched data/steps isolates the vision effect.

Data repairs vs v1.7 (genizah_clean_v2): 45 self-duplicated transcriptions repaired
(keep-gap-marked copy), misaligned image/GT pairings dropped (PGP-metadata ∩ Kraken
screen), digit-boundary-safe decontamination re-verified against the frozen benchmark.

Hard gates (do not launch without): pinned install triplet (audited 2026-08-09:
bug present in unsloth_zoo 2026.8.6, fix valid); cell-5 forward/backward asserting
**all 108** vision lora_B grads nonzero iff the arm trains vision (per-tensor `min`);
cell-7 shipped-adapter check before any push. Colab A100; ~71 min/100 steps measured
(v1.7), expect +15–30% for arm B.


In [ ]:
# Cell 1 — installs + env (PINNED: the exact triplet audited 2026-08-09;
# unpinned installs float and transformers 5.x would invalidate the vision-path
# analysis AND the 108-tensor count)
import os
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
os.environ["WANDB_PROJECT"] = "qwen-hebrew-finetune"
%pip install -q "unsloth[colab-new]==2026.8.9" "unsloth_zoo==2026.8.6" "transformers==4.57.6" hf_transfer wandb

from importlib.metadata import version
assert version("unsloth_zoo") == "2026.8.6", f"unsloth_zoo drifted: {version('unsloth_zoo')}"
assert version("transformers") == "4.57.6", f"transformers drifted: {version('transformers')}"

import torch
assert torch.cuda.is_available(), "No GPU — switch runtime to A100"


In [ ]:
# Cell 2 — data: genizah_clean_v2 (repaired) + talmud v2 + synthetic v2 (all pinned)
from google.colab import userdata
from huggingface_hub import login, snapshot_download
from datasets import load_dataset

login(token=userdata.get("HF_TOKEN"))

GENIZAH_REPO = "isaacmg/genizah_clean_v2"
GENIZAH_REVISION = "57366ad378946918731ad0012d699acc7d9ed31c"  # v2: 1,055/56, repaired+screened
TALMUD_REPO = "isaacmg/talmud_finetune_v2"
SYNTH_REPO = "isaacmg/synthetic_rashi"
SYNTH_REVISION = "b4e1254e1b51ff93958599ccd46ecf2992cd2e97"  # v2: 18-48px renders

assert len(GENIZAH_REVISION) == 40, "pin genizah_clean_v2 to its pushed revision SHA"

genizah = load_dataset(GENIZAH_REPO, split="train", revision=GENIZAH_REVISION)
genizah_val = load_dataset(GENIZAH_REPO, split="val", revision=GENIZAH_REVISION)
talmud = load_dataset(TALMUD_REPO, split="train")
talmud_val = load_dataset(TALMUD_REPO, split="val")
synth = load_dataset(SYNTH_REPO, split="train", revision=SYNTH_REVISION)
synth_eval = load_dataset(SYNTH_REPO, split="eval", revision=SYNTH_REVISION)

crops = talmud.filter(lambda t: t == "crop_transcribe", input_columns="task")
pages = talmud.filter(lambda t: t == "page_extract", input_columns="task")
gemara_crops = crops.filter(lambda s: s == "gemara", input_columns="section")
pages_small = pages.filter(
    lambda s: s in ("rashi", "tosafot"), input_columns="section")
# v1.6 sweep: vols 16 (Nedarim, Ran layout) + 05 host a persistent rashi
# collapse attractor (16_33b) — drop their small-script page rows
# (non-standard layout puts commentary in the wrong margin).
_n_before = len(pages_small)
pages_small = pages_small.filter(
    lambda st: st.split("_")[0] not in ("16", "05"), input_columns="stem")
print(f"pages_small: {_n_before} -> {len(pages_small)} after vol 16/05 exclusion")
assert len(pages_small) < _n_before, "vol 16/05 exclusion filtered nothing"
pages_gemara = pages.filter(lambda s: s == "gemara", input_columns="section")

print(f"genizah={len(genizah)} synth={len(synth)} gemara_crops={len(gemara_crops)} "
      f"pages_small={len(pages_small)} pages_gemara={len(pages_gemara)}")
assert len(genizah) > 1000 and len(genizah_val) >= 50
assert genizah[0]["task"] == "fragment_transcribe"
# label hygiene: no internal gap tokens or mojibake may reach training
_sample_answers = genizah.shuffle(seed=0).select(range(200))["answer"]
assert all("\u2423" not in a for a in _sample_answers), "internal gap token leaked"
assert all(not any(ch in a for ch in "&#$_{}<>\\") for a in _sample_answers), "mojibake/apparatus symbol leaked"
assert any("[...]" in a for a in _sample_answers), "expected damage-gap markers"


In [ ]:
# Cell 3 — model at page resolution + v1.7 step-800 adapter warm start (atomic cell)
from unsloth import FastVisionModel
from transformers import AutoImageProcessor

MAX_SEQ = 12288
# Global 6.5MP floor (train/infer resolution contract, ships in the export).
MIN_PIX = 6_500_000
MAX_PIX = 7_000_000

V17_CKPT_REPO = "isaacmg/qwen3-vl-8b-hebrew-v17-ckpt"
V17_REVISION = "068a2cc1d083ea2fe27b9d09ec801e0cb8474d59"  # step 800, eval_loss 0.7805

# v1.8 A/B protocol — run A first, then B; one variable per step:
#   A (control):   repaired data, vision LoRA frozen (bug left in place)
#   B (candidate): repaired data + vision LoRA actually training
# The warm start's vision adapter is identity (all 108 lora_B == 0 == PEFT
# fresh init), so arm B is equivalent to attaching a brand-new vision adapter
# to v1.7's language state — the warm start cannot contaminate the A/B.
ARM = "A"
assert ARM in ("A", "B"), ARM
TRAIN_VISION_LORA = ARM == "B"

model, tokenizer = FastVisionModel.from_pretrained(
    "unsloth/Qwen3-VL-8B-Instruct",
    load_in_4bit=True,
    use_gradient_checkpointing="unsloth",
    max_seq_length=MAX_SEQ,
)
model = FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers=True, finetune_language_layers=True,
    finetune_attention_modules=True, finetune_mlp_modules=True,
    r=16, lora_alpha=16, lora_dropout=0.0, bias="none", random_state=3407,
)

# weights-only warm start from the v1.7 step-800 adapter (fresh optimizer + schedule)
assert len(V17_REVISION) == 40, "invalid revision SHA"
from safetensors.torch import load_file
from peft import set_peft_model_state_dict
local = snapshot_download(V17_CKPT_REPO, revision=V17_REVISION,
                          allow_patterns="last-checkpoint/adapter_model.safetensors")
missing = set_peft_model_state_dict(
    model, load_file(f"{local}/last-checkpoint/adapter_model.safetensors"))
print("unexpected keys:", len(getattr(missing, "unexpected_keys", [])))

if TRAIN_VISION_LORA:
    # Unsloth's requires_grad_for_gradient_checkpointing never matches the
    # Qwen3-VL vision tower (`enumerate(self.blocks)` loop; forward calls
    # get_image_features, not self.visual), so hidden states entering the
    # checkpointed vision blocks never require grad and UnslothCheckpointFunction
    # skips their backward entirely (re-verified against unsloth_zoo 2026.8.6).
    # Making the patch-embed output require grad restores gradient flow;
    # backward stops cleanly at this leaf. The grad-enabled guard keeps the
    # hook inert inside torch.inference_mode() generation.
    def _vision_embeds_require_grad(module, inputs, output):
        if not torch.is_grad_enabled():
            return output
        output.requires_grad_(True)
        return output
    _patch_embed = next(m for n, m in model.named_modules()
                        if n.endswith("visual.patch_embed"))
    _patch_embed.register_forward_hook(_vision_embeds_require_grad)
    print("vision LoRA gradient fix: ON")

tokenizer.image_processor = AutoImageProcessor.from_pretrained(
    "unsloth/Qwen3-VL-8B-Instruct", min_pixels=MIN_PIX, max_pixels=MAX_PIX,
)
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
lora_params = [n for n, p in model.named_parameters()
               if p.requires_grad and "lora" in n.lower()]
assert trainable > 0 and lora_params
print(f"trainable: {trainable/1e6:.1f}M ({len(lora_params)} LoRA tensors) — "
      "NOTE: requires_grad-based count; the grad-flow cell below is the real check")
print("resolution:", tokenizer.image_processor.size)


In [ ]:
# Cell 4 — conversation format + collator (native res preserved by resize='max')
def to_conversation(sample):
    return {
        "messages": [
            {"role": "user", "content": [
                {"type": "image", "image": sample["image"]},
                {"type": "text", "text": sample["question"]},
            ]},
            {"role": "assistant", "content": [
                {"type": "text", "text": sample["answer"]},
            ]},
        ]
    }

from unsloth.trainer import UnslothVisionDataCollator

collator = UnslothVisionDataCollator(
    model, tokenizer,
    formatting_func=to_conversation,
    resize="max",
    max_seq_length=MAX_SEQ,
    train_on_responses_only=True,
    instruction_part="<|im_start|>user\n",
    response_part="<|im_start|>assistant\n",
)

# guardrail: one real batch must show page-res pixels + masked labels
batch = collator([genizah[0], synth[0]])
pv = batch["pixel_values"]
assert pv is not None and pv.shape[0] > 40000, (
    f"{pv.shape[0]} patch rows — expected >40k for two ~6.5MP images; "
    "the resolution policy is not reaching the collator")
labels = batch["labels"]
unmasked = labels[0][labels[0] != -100]
_tok = tokenizer.tokenizer if hasattr(tokenizer, "tokenizer") else tokenizer
assert genizah[0]["answer"][:30] in _tok.decode(unmasked)
print(f"collator OK: pixel rows={pv.shape[0]}, labels masked")


In [ ]:
# Cell 5 — gradient-flow verification (cheap; run BEFORE burning GPU-hours)
# One forward/backward on a real genizah sample. Language LoRA must always
# receive gradients; vision LoRA must receive them iff TRAIN_VISION_LORA.
# HARD GATE: per-tensor (min) nonzero across all 108 vision lora_B when ON.
FastVisionModel.for_training(model)
_vb = {k: (v.to(model.device) if torch.is_tensor(v) else v)
       for k, v in collator([genizah[0]]).items()}
model(**_vb).loss.backward()
vis_g = [p.grad.abs().max().item() for n, p in model.named_parameters()
         if ".visual." in n and "lora_B" in n and p.grad is not None]
lang_g = [p.grad.abs().max().item() for n, p in model.named_parameters()
          if ".visual." not in n and "lora_B" in n and p.grad is not None]
model.zero_grad(set_to_none=True)
assert lang_g and max(lang_g) > 0, "language LoRA got no gradients"
if TRAIN_VISION_LORA:
    assert len(vis_g) == 108 and min(vis_g) > 0, (
        f"vision LoRA dead or partial: {len(vis_g)}/108 tensors with grads, "
        f"min|g|={min(vis_g) if vis_g else 0:.2e}")
    print(f"vision LoRA training CONFIRMED: 108/108 grads, "
          f"min|g|={min(vis_g):.2e} max|g|={max(vis_g):.2e}")
else:
    assert not vis_g or max(vis_g) == 0, (
        "vision grads present but flag is False — a newer unsloth_zoo may have "
        "fixed hook discovery upstream; re-audit before trusting this arm")
    print("language-only training confirmed (vision LoRA frozen, as configured)")
print(f"language max|g|={max(lang_g):.2e}")


In [ ]:
# Cell 6 — mixture + per-domain eval + training (resumable, max_steps-bounded)
import inspect
import wandb
from datasets import concatenate_datasets, interleave_datasets
from huggingface_hub import list_repo_files
from trl import SFTConfig, SFTTrainer
from unsloth import is_bf16_supported

CKPT_REPO = f"isaacmg/qwen3-vl-8b-hebrew-v18{ARM.lower()}-ckpt"  # private, auto-created
OUT_DIR = f"outputs_v18{ARM.lower()}"
wandb.login(key=userdata.get("WANDB_API_KEY"))

# 35% genizah / 25% small-script pages / 20% synth / 15% gemara crops / 5% gemara pages
mixture = interleave_datasets(
    [genizah, pages_small, synth, gemara_crops, pages_gemara],
    probabilities=[0.35, 0.25, 0.20, 0.15, 0.05], seed=3407,
    stopping_strategy="all_exhausted",
)
# val loss every 100 steps: genizah + talmud pages + gemara crops + synth
eval_ds = concatenate_datasets([
    genizah_val.select(range(40)),
    talmud_val.filter(lambda t: t == "page_extract", input_columns="task")
              .select(range(40)),
    talmud_val.filter(
        lambda t, s: t == "crop_transcribe" and s == "gemara",
        input_columns=["task", "section"]).select(range(20)),
    synth_eval.select(range(20)),
])

def make_sft_config(**kw):
    params = inspect.signature(SFTConfig.__init__).parameters
    if "max_seq_length" in kw and "max_seq_length" not in params:
        kw["max_length"] = kw.pop("max_seq_length")
    dropped = {k: kw.pop(k) for k in list(kw) if k not in params}
    if dropped:
        print(f"⚠️ dropped unsupported SFTConfig kwargs: {sorted(dropped)}")
    return SFTConfig(**kw)

def make_trainer(**kw):
    try:
        return SFTTrainer(**kw)
    except TypeError as e:
        if "tokenizer" in kw and ("tokenizer" in str(e) or "processing_class" in str(e)):
            kw["processing_class"] = kw.pop("tokenizer")
            return SFTTrainer(**kw)
        raise

resume_dir = None
try:
    # hub_strategy="checkpoint" pushes a rolling "last-checkpoint/" folder
    files = list_repo_files(CKPT_REPO)
    if any(f.startswith("last-checkpoint/") for f in files):
        snapshot_download(CKPT_REPO, allow_patterns="last-checkpoint/*",
                          local_dir=OUT_DIR)
        resume_dir = f"{OUT_DIR}/last-checkpoint"
        print("resuming from last-checkpoint")
except Exception as e:
    print(f"no checkpoint repo yet ({type(e).__name__}) — fresh start")

FastVisionModel.for_training(model)
trainer = make_trainer(
    model=model, tokenizer=tokenizer, data_collator=collator,
    train_dataset=mixture, eval_dataset=eval_ds,
    args=make_sft_config(
        per_device_train_batch_size=1, gradient_accumulation_steps=8,
        max_steps=700,                     # completes v1.7's 800/1500 schedule
        learning_rate=5e-5,                # continuation LR
        warmup_ratio=0.02, lr_scheduler_type="cosine", weight_decay=0.01,
        logging_steps=10,
        eval_strategy="steps", eval_steps=100, per_device_eval_batch_size=1,
        save_steps=100, save_total_limit=2,
        push_to_hub=True, hub_model_id=CKPT_REPO,
        hub_strategy="checkpoint", hub_private_repo=True,
        optim="adamw_8bit", seed=3407, output_dir=OUT_DIR,
        report_to="wandb", run_name=f"genizah_focus_v18{ARM.lower()}",
        bf16=is_bf16_supported(), fp16=not is_bf16_supported(),
        remove_unused_columns=False, dataset_text_field="",
        dataset_kwargs={"skip_prepare_dataset": True},
        max_seq_length=MAX_SEQ,
    ),
)
trainer.train(resume_from_checkpoint=resume_dir)


In [ ]:
# Cell 7 — export merged (policy-carrying) model, gated on the vision check
MERGED_REPO = f"isaacmg/qwen3-vl-8b-hebrew-v18{ARM.lower()}-merged"
if TRAIN_VISION_LORA:
    # shipped-adapter gate: the run is not a vision run unless the weights moved
    _nz = [n for n, p in model.named_parameters()
           if ".visual." in n and "lora_B" in n and p.detach().abs().max().item() > 0]
    assert len(_nz) == 108, f"shipped adapter vision lora_B nonzero: {len(_nz)}/108"
    print("shipped-adapter vision check: 108/108 lora_B nonzero")
model.save_pretrained_merged("v18-merged", tokenizer, save_method="merged_16bit")
# ship the TRAINING resolution policy with the model (hard rule)
tokenizer.image_processor.save_pretrained("v18-merged")
model.push_to_hub_merged(MERGED_REPO, tokenizer, save_method="merged_16bit", private=True)
print("pushed", MERGED_REPO)
